# Customer Sentiment Analyzer
## VADER + TF-IDF + Logistic Regression


### 1. Imports and dataset


In [19]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer


In [20]:
nltk.download("vader_lexicon")

df = pd.read_csv("../data/raw/customer_messages.csv")
print("Dataset shape:", df.shape)


Dataset shape: (300, 1)


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sy377\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### 2. VADER baseline


In [21]:
sia = SentimentIntensityAnalyzer()

def get_vader_score(text):
    scores = sia.polarity_scores(text)
    return scores["compound"]

def get_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["vader_score"] = df["message"].apply(get_vader_score)
df["vader_sentiment"] = df["vader_score"].apply(get_sentiment)

df.to_csv("../data/processed/vader_results.csv", index=False)
df[["message", "vader_score", "vader_sentiment"]].head(10)


,message,vader_score,vader_sentiment
0,Customer support replied to my message.,0.4019,Positive
1,I received a damaged product.,-0.4404,Negative
2,I appreciate how quickly customer support resp...,0.6597,Positive
3,The product is useful and performs very well.,0.6478,Positive
4,I had a wonderful shopping experience.,0.5719,Positive
5,I was charged incorrectly for the order.,-0.2023,Negative
6,The item arrived without a delay.,0.2411,Positive
7,I have used the product for a few days.,0.0000,Neutral
8,Very satisfied with both the product and deliv...,0.4754,Positive
9,The package was badly damaged.,-0.7184,Negative


### 3. TF-IDF feature extraction


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(df["message"])
y = df["vader_sentiment"]

print("TF-IDF shape:", X.shape)


TF-IDF shape: (300, 1317)


### 4. Train/Test split — 240 training, 60 testing


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(y_train))
print("Testing messages:", len(y_test))


Training messages: 240
Testing messages: 60


### 5. Balanced Logistic Regression


In [24]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


### 6. Model evaluation


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

report = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))


Accuracy: 71.67%
              precision    recall  f1-score   support

    Negative       1.00      0.31      0.47        13
     Neutral       0.71      0.91      0.80        22
    Positive       0.68      0.76      0.72        25

    accuracy                           0.72        60
   macro avg       0.80      0.66      0.66        60
weighted avg       0.76      0.72      0.69        60



In [26]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Negative", "Neutral", "Positive"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Neutral", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Neutral", "Predicted Positive"]
)

cm_df


,Predicted Negative,Predicted Neutral,Predicted Positive
Actual Negative,4,2,7
Actual Neutral,0,20,2
Actual Positive,0,6,19


### 7. VADER vs ML comparison on the 60 test messages


In [27]:
comparison = df.loc[y_test.index].copy()
comparison["ml_prediction"] = y_pred
comparison["match"] = comparison["vader_sentiment"] == comparison["ml_prediction"]

match_count = comparison["match"].sum()
total = len(comparison)

print("Matching predictions:", match_count)
print("Total test messages:", total)
print("Agreement:", f"{match_count / total * 100:.2f}%")

comparison[["message", "vader_score", "vader_sentiment", "ml_prediction", "match"]].head(20)


Matching predictions: 43
Total test messages: 60
Agreement: 71.67%


,message,vader_score,vader_sentiment,ml_prediction,match
194,The delivery took about the stated number of d...,0.0772,Positive,Neutral,False
277,The support response contained the information...,0.4019,Positive,Positive,True
83,I am not satisfied with the product.,-0.3252,Negative,Positive,False
74,The quality is worse than I expected.,-0.4767,Negative,Positive,False
244,I received poor customer service.,-0.4767,Negative,Negative,True
208,The order status was visible in the account.,0.0000,Neutral,Neutral,True
216,The product performance is poor.,-0.4767,Negative,Negative,True
36,The item looks different from the listing.,0.0000,Neutral,Neutral,True
203,The quality and performance are excellent.,0.5719,Positive,Positive,True
215,I am impressed by the quick delivery.,0.4767,Positive,Positive,True


### 8. Save evaluation results for the Admin dashboard


In [28]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

with open("../data/processed/classification_report.json", "w") as f:
    json.dump(report, f, indent=4)

cm_df.to_csv("../data/processed/confusion_matrix.csv")

with open("../data/processed/accuracy.txt", "w") as f:
    f.write(str(accuracy))

comparison.to_csv("../data/processed/comparison_results.csv", index=False)

print("Evaluation files saved successfully.")


Evaluation files saved successfully.


### 9. Save the final model for the project
The evaluation above uses 240 training messages and 60 test messages. After evaluation, the final deployment model is trained on all 300 VADER-labeled messages.


In [29]:
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

final_model.fit(X, y)

joblib.dump(final_model, "../models/sentiment_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("Final model and TF-IDF vectorizer saved successfully.")


Final model and TF-IDF vectorizer saved successfully.


### 10. Live review test


In [30]:
test_review = "The product is very bad and I am very sad."
score = sia.polarity_scores(test_review)
print(score)
print("VADER sentiment:", get_sentiment(score["compound"]))


{'neg': 0.507, 'neu': 0.493, 'pos': 0.0, 'compound': -0.8012}
VADER sentiment: Negative


### Model configuration used
- Baseline: VADER Lexicon
- Feature extraction: TF-IDF with unigram + bigram features
- ML algorithm: Logistic Regression
- Class weighting: balanced
- Dataset: 300 customer messages
- Evaluation: 240 training + 60 testing
- Customer-facing live sentiment: VADER


In [31]:
print("Accuracy:", accuracy)
print("Training messages:", len(y_train))
print("Testing messages:", len(y_test))
print("Model:", model)
print("Vectorizer:", vectorizer)

Accuracy: 0.7166666666666667
Training messages: 240
Testing messages: 60
Model: LogisticRegression(class_weight='balanced', max_iter=1000)
Vectorizer: TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)


In [33]:
import sys
import os

BASE_DIR = os.path.dirname(os.getcwd())
sys.path.append(BASE_DIR)

from src.ml_model import create_vectorizer, create_model

vectorizer_test = create_vectorizer()
model_test = create_model()

print(vectorizer_test)
print(model_test)

TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
LogisticRegression(class_weight='balanced', max_iter=1000)


In [36]:
from src.ml_model import train_model, predict
from sklearn.model_selection import train_test_split

messages = df["message"]
labels = df["vader_sentiment"]

train_messages, test_messages, train_labels, test_labels = train_test_split(
    messages,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

vectorizer_src, model_src = train_model(
    train_messages,
    train_labels
)

y_pred_src = predict(
    test_messages,
    vectorizer_src,
    model_src
)

print("Model trained successfully.")
print("Training messages:", len(train_messages))
print("Testing messages:", len(test_messages))
print("Predictions:", len(y_pred_src))

Model trained successfully.
Training messages: 240
Testing messages: 60
Predictions: 60


In [38]:
from sklearn.metrics import accuracy_score

src_accuracy = accuracy_score(test_labels, y_pred_src)

print(f"src Model Accuracy: {src_accuracy * 100:.2f}%")

src Model Accuracy: 73.33%


In [39]:
print("Notebook X_train shape:", X_train.shape)
print("Notebook X_test shape:", X_test.shape)
print("src train size:", len(train_messages))
print("src test size:", len(test_messages))

Notebook X_train shape: (240, 1317)
Notebook X_test shape: (60, 1317)
src train size: 240
src test size: 60


In [40]:
print("Notebook accuracy:", accuracy_score(y_test, y_pred))
print("src accuracy:", src_accuracy)

Notebook accuracy: 0.7166666666666667
src accuracy: 0.7333333333333333


In [41]:
print("Notebook TF-IDF features:", X_train.shape[1])
print("SRC TF-IDF features:", vectorizer_src.transform(train_messages).shape[1])

Notebook TF-IDF features: 1317
SRC TF-IDF features: 1130


In [42]:
import os
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from src.ml_model import train_model, predict

messages = df["message"]
labels = df["vader_sentiment"]

train_messages, test_messages, train_labels, test_labels = train_test_split(
    messages,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

vectorizer, model = train_model(
    train_messages,
    train_labels
)

y_pred = predict(
    test_messages,
    vectorizer,
    model
)

accuracy = accuracy_score(test_labels, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Training messages: {len(train_messages)}")
print(f"Testing messages: {len(test_messages)}")

report = classification_report(
    test_labels,
    y_pred,
    output_dict=True
)

cm = confusion_matrix(
    test_labels,
    y_pred,
    labels=["Negative", "Neutral", "Positive"]
)

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

with open("../data/processed/classification_report.json", "w") as f:
    json.dump(report, f, indent=4)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Neutral", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Neutral", "Predicted Positive"]
)

cm_df.to_csv("../data/processed/confusion_matrix.csv")

with open("../data/processed/accuracy.txt", "w") as f:
    f.write(str(accuracy))

joblib.dump(model, "../models/sentiment_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("Final model and evaluation files saved successfully.")

Accuracy: 73.33%
Training messages: 240
Testing messages: 60
Final model and evaluation files saved successfully.


In [43]:
comparison = df.loc[test_messages.index].copy()

comparison["ml_prediction"] = y_pred
comparison["match"] = (
    comparison["vader_sentiment"] == comparison["ml_prediction"]
)

comparison.to_csv(
    "../data/processed/comparison_results.csv",
    index=False
)

print("Comparison results saved successfully.")
print(f"Matches: {comparison['match'].sum()}/{len(comparison)}")

Comparison results saved successfully.
Matches: 44/60
